In [17]:
import warnings
warnings.filterwarnings('ignore')

import calitp_data_analysis.magics

In [18]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
# from siuba import *

fs = get_fs()

In [29]:
import altair as alt
import folium
import matplotlib.pyplot as plt



ModuleNotFoundError: No module named 'mapclassify'

In [20]:
from IPython.display import HTML

In [21]:
pd.set_option("display.max_columns", None)

In [6]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"

In [7]:
shape_data_name = "replica-stm_scag_origins-origin_layer.zip"

origins_name = "replica-stm_mcog_origins_origin_layer.zip"
shape_destinations_name = "replica-stm_mcog_destinations-destination_layer.zip"

#### Check Shape Data -- Hex Cells

* When you create a study in Replica, you can download the geography breakdown that is available within the map view
* We want to change that geography breakdown to `H3 Hex Cells Resolution 7 (~2 sq mi/cell)`
* Set up your analysis as stated in the Analysis Steps
* Download the Hex Cells once you have the top 50 trip hex cells
* Upload to GCS and follow the steps below before uploading to Replica as a Custom Geographies

In [8]:
# with get_fs().open(f"{gcs_path}{shape_data_name}") as f:
#         shps = to_snakecase(gpd.read_file(f))


with get_fs().open(f"{gcs_path}origins/{origins_name}") as f:
        shps = to_snakecase(gpd.read_file(f))

In [9]:
with get_fs().open(f"{gcs_path}origins/{shape_destinations_name}") as f:
        shps = to_snakecase(gpd.read_file(f))

In [10]:
assert (len(shps) == 50)

In [11]:
shps.sample()

,geoname,customgeo,centrlat,centrlon,sqmi,destin,dest_sqmi,geometry
44,872806d4effffff,608690106613628900,39.3773,-123.337,2.0104,189,94.0099,"POLYGON ((-123.32800 39.36710, -123.34446 39.3..."


In [12]:
shps = shps.set_geometry("geometry")

In [13]:
# (shps.sample()).explore()

* Rename the columns to match Replica's Custom Geographies specs

In [14]:
shps = shps.rename(columns={"geoname":"name", "customgeo":"id"})

In [15]:
shps.sample()

,name,id,centrlat,centrlon,sqmi,destin,dest_sqmi,geometry
14,872806d2cffffff,608690106043203600,39.2454,-123.2037,2.0166,768,380.8321,"POLYGON ((-123.19473 39.23511, -123.21118 39.2..."


In [16]:
geojson_string = shps.to_json()

* Run the cell below `geojson_string`
* Copy and paste the string in a new file
* Rename the file "Shps_{County}_Origins.geojson"
* Download the new geojson and upload to Replica Custom Geographies
* Use the newly uploaded Geography as the custom geography for the Trip Data Download (steps detailed in Analysis Steps)

In [17]:
geojson_string

'{"type": "FeatureCollection", "features": [{"id": "0", "type": "Feature", "properties": {"name": "8728316abffffff", "id": 608693033046638600, "centrlat": 39.1355, "centrlon": -123.1962, "sqmi": 2.0197, "destin": 5419, "dest_sqmi": 2683.106}, "geometry": {"type": "Polygon", "coordinates": [[[-123.187240573, 39.125197849], [-123.203678209, 39.123788041], [-123.212646512, 39.134077515], [-123.205177356, 39.145777898], [-123.188736333, 39.147188861], [-123.179767853, 39.136898286], [-123.187240573, 39.125197849]]]}}, {"id": "1", "type": "Feature", "properties": {"name": "87283168dffffff", "id": 608693032543322100, "centrlat": 39.1575, "centrlon": -123.1977, "sqmi": 2.0191, "destin": 3562, "dest_sqmi": 1764.1803}, "geometry": {"type": "Polygon", "coordinates": [[[-123.188736333, 39.147188861], [-123.205177356, 39.145777898], [-123.21414767, 39.156064585], [-123.206677139, 39.167763335], [-123.190232729, 39.169175453], [-123.181262237, 39.158887665], [-123.188736333, 39.147188861]]]}}, {"id

In [18]:
# saved = to_snakecase(gpd.read_file("./saved-selection-geo-stm-scag-top50-origins_h3z7.geojson"))


In [19]:
# saved

##### Checking to see if geonames match

In [20]:
shps[shps['id']==608718595349807100]

,name,id,centrlat,centrlon,sqmi,destin,dest_sqmi,geometry


#### Check Trips Data

In [34]:
trips = "replica-stm_mtc_origins-06_11_26-trips_dataset.zip"

In [35]:
df = to_snakecase(pd.read_csv(f"{gcs_path}{trips}"))  

In [36]:
df.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,tour_type,trip_start_local_hour,trip_end_local_hour,origin_custom_id,destination_custom_id,origin_zcta_2020,destination_zcta_2020,destination_custom_lat,destination_custom_lng,origin_custom_lng,origin_custom_lat
494816,11450995759283825128,"1 (Tract 9821, Alameda, CA)","9821 (Alameda, CA)","Alameda County, CA",California,"2 (Tract 611.01, San Francisco, CA)","611.01 (San Francisco, CA)","San Francisco County, CA",California,biking,recreation,recreation,04:30:00,05:59:15,89,15.1,unknown_vehicle_type,unknown_fuel_type,NaN,NaN,NaN,non_retail_attraction,non_retail_attraction,retail,retail,17490821120225364441,17872052184154803393,68.0,male,black_not_hispanic_or_latino,not_in_labor_force,unemployed_under_16_not_in_labor_force,5911.0,other_travel_mode,1,5911.0,zero,core,not_working,single_family,not_attending_school,some_college,owner,english,"4 (Tract 4219, Alameda, CA)","4219 (Alameda, CA)","Alameda County, CA",California,Does not have work/school location,Does not have work/school location,Does not have work/school location,Does not have work/school location,8728308a8ffffff,87283082affffff,other_home_based,4,5,608692972866764800,6.086930e+17,94720,94111,37.7865,-122.3945,-122.2513,37.8646


In [37]:
len(df.origin_custom_id.value_counts())

50

In [38]:
# df.columns

In [39]:
df.destination_custom_id.nunique()

8254

In [40]:
len(df[df['destination_custom_id']==None])

0

In [42]:
# df.destination_custom_id.info()

In [27]:
with get_fs().open(f"{gcs_path}origins/replica-stm_hex_cells-origin_layer.zip") as f:
        hex_cells = to_snakecase(gpd.read_file(f))

In [28]:
hex_cells.sample()

,geoname,customgeo,centrlat,centrlon,sqmi,origin,orig_sqmi,geometry
20265,872989358ffffff,608716658269225000,39.7014,-120.3176,2.0577,8,3.8879,"POLYGON ((-120.30870 39.69083, -120.32555 39.6..."


In [29]:
hex_cells = hex_cells.set_geometry("geometry")

In [30]:
hex_cells = hex_cells.rename(columns={"geoname":"name", "customgeo":"id"})

In [31]:
hex_geojson_string = hex_cells.to_json()

In [32]:
# hex_geojson_string

In [33]:
# with open("hex7_layer.geojson", "w") as file:
#     file.write(hex_geojson_string)

In [33]:
utils.make_zipped_shapefile(hex_cells, "hex7_layer.zip")

Path name: hex7_layer.zip
Dirname (1st element of path): hex7_layer
Shapefile name: hex7_layer.shp
Shapefile component parts folder: hex7_layer/hex7_layer.shp


In [22]:
hexcells = to_snakecase(gpd.read_file("hex7_layer.zip"))

In [23]:
hexcells.sample()

,name,id,centrlat,centrlon,sqmi,origin,orig_sqmi,geometry
26770,87283370effffff,608693172146536400,39.5543,-122.1255,2.0296,3,1.4781,"POLYGON ((-122.11653 39.54390, -122.13316 39.5..."


In [24]:
len(hexcells)

34623

In [30]:
hexcells = hexcells.set_geometry("geometry")

In [39]:
# (hexcells.sample(50)).explore()

In [33]:
# ! pip install folium matplotlib mapclassify